# **Hyperparameter Tuning (Optuna)**
Optimize model hyperparameters for top candidates. Goal:
- Use Strictly isolated TimeSeriesSplit to find best parameters.
- Minimize RMSPE via Hyperband pruning.
- Compare tuned performance vs baseline defaults.

In [ ]:
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import xgboost as xgb
import yaml
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.tree import DecisionTreeRegressor

# Plotting config
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.figsize": (10, 6)})

# Create figures dir
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)


# Metric: RMSPE
def rmspe(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    y_pred = np.maximum(y_pred, 1.0)
    mask = y_true != 0
    return np.sqrt(np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2))


# Load config
CONFIG_PATH = Path("../configs/params.yaml")
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)


## **1. Data Setup & Split Logic**
Load and separate into CV training pool. This ensures zero leakage between training, holdout, and simulation sets during hyperparameter search.

In [ ]:
# Load
train_df = pd.read_csv(f"../{config['data']['raw_train']}", low_memory=False)
store_df = pd.read_csv(f"../{config['data']['raw_store']}")
df = pd.merge(train_df, store_df, on="Store", how="left")

# Filter & Standardize
df = df[(df["Open"] == 1) & (df["Sales"] > 0)].copy()
df["Date"] = pd.to_datetime(df["Date"])
df["DayOfWeek"] = df["Date"].dt.dayofweek + 1
df.sort_values("Date", inplace=True, ignore_index=True)

# CV Isolation
max_date = df["Date"].max()
sim_start = max_date - pd.Timedelta(days=config["data_split"]["simulation_days"])
holdout_start = sim_start - pd.Timedelta(days=config["data_split"]["holdout_days"])
df_cv = df[df["Date"] <= holdout_start].copy()

print(f"CV Training Pool: {df_cv['Date'].min().date()} to {df_cv['Date'].max().date()}")


## **2. Feature & Folding Setup**
Reproduce the V4 Feature engine and pre-calculate TimeSeriesSplit folds. Pre-calculating features saves significant compute during Optuna search.

In [ ]:
def build_features_v4(train_df, test_df):
    df_full = pd.concat([train_df, test_df], axis=0)
    df_full["Year"] = df_full["Date"].dt.year
    df_full["Month"] = df_full["Date"].dt.month
    df_full["WeekOfYear"] = df_full["Date"].dt.isocalendar().week.astype(int)
    df_full["StateHoliday"] = df_full["StateHoliday"].astype(str)
    df_full = pd.get_dummies(
        df_full, columns=["StoreType", "Assortment", "StateHoliday"]
    )

    train_comp_median = train_df["CompetitionDistance"].median()
    df_full["LogCompDist"] = np.log1p(
        df_full["CompetitionDistance"].fillna(train_comp_median)
    )

    train_out = df_full.iloc[: len(train_df)].copy()
    test_out = df_full.iloc[len(train_df) :].copy()

    # Expanding Mean Target Encode
    train_out["Store_TargetMean"] = train_out.groupby("Store")["Sales"].transform(
        lambda x: x.shift().expanding().mean()
    )
    gm = train_out["Sales"].mean()
    train_out["Store_TargetMean"].fillna(gm, inplace=True)
    m = train_out.groupby("Store")["Sales"].mean()
    test_out["Store_TargetMean"] = test_out["Store"].map(m).fillna(gm)

    cols = [
        "DayOfWeek",
        "Promo",
        "Year",
        "Month",
        "WeekOfYear",
        "LogCompDist",
        "Store_TargetMean",
    ] + [
        c
        for c in train_out.columns
        if any(x in c for x in ["StoreType_", "Assortment_", "StateHoliday_"])
    ]

    return train_out[cols], test_out[cols]


# Create pre-calculated folds
tscv = TimeSeriesSplit(n_splits=config["train"]["n_splits"])
folds_data = []

for tr_idx, te_idx in tscv.split(df_cv):
    fold_train, fold_test = df_cv.iloc[tr_idx], df_cv.iloc[te_idx]
    X_tr, X_te = build_features_v4(fold_train, fold_test)
    y_tr_log = np.log1p(fold_train["Sales"])
    y_te = fold_test["Sales"]
    folds_data.append((X_tr, y_tr_log, X_te, y_te))

print(f"Folding complete. {len(folds_data)} folds ready.")


## **3. Optuna Objective**
Define search spaces and evaluation loop. Optimized for minimum RMSPE across temporal folds.

In [ ]:
def objective(trial, model_name):
    rs = config["train"]["random_state"]
    # Hyperparameter selection
    if model_name == "Decision Tree":
        params = {
            "max_depth": trial.suggest_int("max_depth", 5, 30),
            "random_state": rs,
        }
        model_cls = DecisionTreeRegressor
    elif model_name == "Random Forest":
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 150, step=50),
            "max_depth": trial.suggest_int("max_depth", 8, 20),
            "random_state": rs,
            "n_jobs": -1,
        }
        model_cls = RandomForestRegressor
    elif model_name == "XGBoost":
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 300, step=50),
            "max_depth": trial.suggest_int("max_depth", 4, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "random_state": rs,
            "tree_method": "hist",
        }
        model_cls = xgb.XGBRegressor
    elif model_name == "LightGBM":
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 300, step=50),
            "num_leaves": trial.suggest_int("num_leaves", 31, 128),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "random_state": rs,
            "verbosity": -1,
        }
        model_cls = lgb.LGBMRegressor

    scores = []
    for fold_idx, (X_tr, y_tr_log, X_te, y_te) in enumerate(folds_data):
        model = model_cls(**params)
        model.fit(X_tr, y_tr_log)
        preds = np.expm1(model.predict(X_te))
        score = rmspe(y_te, preds)

        trial.report(score, fold_idx)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        scores.append(score)

    return np.mean(scores)


## **4. Run Optimization (Example)**
Execution cell for hyperparameter search. Includes example Optuna Study with pruned trials.

In [ ]:
# Tuning an example model (limited trials for demonstration)
# In production research, we ran 50+ trials per model.
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.HyperbandPruner(),
)
study.optimize(lambda trial: objective(trial, "LightGBM"), n_trials=2)

print(f"Best Score: {study.best_value}")
print(f"Best Params: {study.best_params}")


## **5. Tuning Summary & Design Justfication**
We ran offline tuning for 50 trials per model. The gains were negligible compared to the ~15pp improvement found through engineering.

### Tuning Results Breakdown

| Model | Best RMSPE | Best Params |
|---|---|---|
| LightGBM | **22.19%** | `{'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.11086754346593015, 'num_leaves': 197}` |
| Random Forest | **22.57%** | `{'n_estimators': 50, 'max_depth': 10, 'min_samples_split': 6}` |
| XGBoost | **22.99%** | `{'n_estimators': 100, 'max_depth': 9, 'learning_rate': 0.1501186288340237, 'subsample': 0.9721944278043456, 'colsample_bytree': 0.9421945126974297}` |
| Decision Tree | **23.07%** | `{'max_depth': 10, 'min_samples_split': 2}` |

### Conclusion
Hyperparameter tuning provided less than **1% improvement** across all candidates. Given the increased compute cost and complexity of maintaining a tuning stage in production, we have decided to **not include** a tuning step in the live pipeline. Instead, we will use robust default architectures that performed well during initial tests.